# 5-1 PEFT(파라미터 효율적 튜닝) 과제

---

## 학습 개요

### 학습 주제
- **Text-to-SQL 태스크에 PEFT 적용**: 자연어 질문을 SQL 쿼리로 변환하는 태스크에 LoRA 기반 파인튜닝을 적용합니다.
- **Chat Template과 Label Masking**: instruction tuning에서 중요한 입력 형식과 학습 대상 토큰 마스킹 기법을 이해합니다.
- **LoRA 하이퍼파라미터 실험**: rank, alpha, target_modules 설정에 따른 학습 효과를 비교합니다.

### 학습 목표
- LoRA 기반 PEFT를 Text-to-SQL 태스크에 적용하고, **chat template 구성과 label masking**을 직접 구현하는 것을 목표로 합니다.
  - Text-to-SQL 태스크의 입출력 형식을 이해하고 **데이터를 변환**할 수 있다.
  - Chat template을 구성하고 label masking(-100)의 원리를 **설명**할 수 있다.
  - LoRA 하이퍼파라미터(r, alpha)를 변경하며 학습 효과를 **비교**할 수 있다.
  - 학습된 모델로 SQL 쿼리를 **생성**하고 결과를 **검증**할 수 있다.

### 핵심 개념
- **Text-to-SQL**: 자연어 질문을 SQL 쿼리로 변환하는 NLP 태스크입니다. "2023년 매출이 가장 높은 제품은?" → `SELECT product FROM sales WHERE year=2023 ORDER BY revenue DESC LIMIT 1`과 같이 변환합니다.
- **Chat Template**: LLM에 입력을 전달할 때 사용하는 표준화된 형식입니다. system/user/assistant 역할을 구분하여 모델이 대화 맥락을 이해할 수 있게 합니다. 각 모델마다 고유한 특수 토큰(예: `<start_of_turn>`, `<end_of_turn>`)을 사용합니다.
- **Label Masking**: instruction tuning에서 입력(instruction) 부분은 학습하지 않고 출력(response) 부분만 학습하도록 레이블을 -100으로 마스킹하는 기법입니다. CrossEntropyLoss는 -100 레이블을 무시하므로, 모델이 "답변 생성"에만 집중하도록 합니다.

### 선행 지식
- **5-1 PEFT 실습 완료 필수**: LoRA 원리, Unsloth 사용법, SFTTrainer 이해
- **SQL 기초**: SELECT, WHERE, ORDER BY 등 기본 쿼리 문법
- **PyTorch 기초**: 텐서 연산, 모델 추론

### 실습-과제 연결
- **실습에서 배운 것**: LoRA 원리, QLoRA 적용, SFTTrainer 사용, 모델 저장/추론
- **과제에서 적용할 것**: 다른 태스크(Text-to-SQL)에 동일 기법 적용, chat template 직접 구현, label masking 이해
- **차이점**: 실습은 체스 기보 데이터, 과제는 Text-to-SQL 데이터 사용

### Further Reading
* [Unsloth 공식 문서](https://docs.unsloth.ai/): Unsloth 프레임워크의 상세 사용법과 최적화 팁
* [LoRA 논문](https://arxiv.org/abs/2106.09685): Low-Rank Adaptation의 원리와 수학적 배경
* [Spider 데이터셋](https://yale-lily.github.io/spider): Text-to-SQL 벤치마크 데이터셋 정보

### Open-ended Mission
- 다양한 rank(r) 값으로 실험하고 파라미터 수 대비 성능 변화를 분석해보세요.
- 생성된 SQL 쿼리의 문법적 정확성을 검증하는 방법을 고민해보세요.

---

## 과제 구성

### 학습 방향

- **과제 구성 방식**
  - 실습에서 배운 LoRA 개념을 다른 프레임워크/태스크에 적용합니다. 코드 틀이 제공되며, 학습자는 설정값 변경 + 결과 확인에 집중합니다.

- **Required Package**
  - `transformers>=4.56.0`: HuggingFace 모델/토크나이저
  - `peft>=0.17.0`: Parameter-Efficient Fine-Tuning 라이브러리
  - `trl>=0.22.0`: Transformer Reinforcement Learning (SFTTrainer)
  - `torch>=2.0.0`: PyTorch 딥러닝 프레임워크
  - `datasets>=4.0.0`: HuggingFace 데이터셋 라이브러리
  - `accelerate>=1.0.0`: 분산 학습 및 혼합 정밀도 지원
  - `numpy>=2.0.0`: 수치 연산 라이브러리
  - `pandas>=2.0.0`: 데이터 분석 라이브러리

- **Step 요약**
  - **Step 1 (25분)**: 데이터셋 불러오기 및 EDA — 데이터 특성 파악
  - **Step 2 (15분)**: 모델/토크나이저 로드 — HuggingFace 모델 로드
  - **Step 3 (40분)**: 데이터 전처리 — chat template, label masking 적용
  - **Step 4 (40분)**: LoRA 학습 — LoRA Config, SFTTrainer 설정 및 학습

- **데이터셋 개요 및 저작권 정보**
  - **데이터셋 명**: shangrilar/ko_text2sql
  - **데이터셋 개요**: 한국어 자연어 질문과 SQL 쿼리 쌍으로 구성된 데이터셋으로, context(스키마), question(질문), answer(SQL) 형태입니다.
  - **사용 목적**: Text-to-SQL 태스크 LoRA 파인튜닝
  - **저작권/출처**: HuggingFace Hub (cc-by-4.0)
  - **주의사항**: 본 과제에서는 빠른 실습을 위해 100 step만 학습합니다. 전체 학습 시 데이터셋의 토큰 길이가 모델의 max_position_embeddings를 초과하지 않는지 확인하세요.

### 문제 설명

- **문제 개요**: 이 과제는 **Text-to-SQL 태스크에 LoRA fine-tuning을 적용**합니다. 학습자는 실습에서 배운 LoRA 개념을 활용하여 **표준 HuggingFace PEFT로 다른 태스크를 학습**할 수 있어야 합니다.
- **요구사항 요약**
  - **EDA**: 데이터셋의 sequence length 분석 및 max_seq_length 결정
  - **데이터 변환**: chat template 적용 및 텍스트 변환
  - **LoRA 설정**: r, lora_alpha, target_modules 파라미터 설정
  - **학습 설정**: batch_size, gradient_accumulation_steps 설정

### 학습 문제

- **Step 1 — 데이터셋 불러오기 및 EDA**
  - **TODO 1**: max_sequence_length 확인 *(연결: EDA / 데이터 분석)*
  - **1줄 요약**: 각 컬럼(context, question, answer)의 최대 문자열 길이를 계산한다.
- **Step 3 — 데이터 전처리**
  - **TODO 2**: convert_to_conversation 함수 작성 *(연결: chat template / 데이터 포맷팅)*
  - **1줄 요약**: 데이터를 system/user/assistant messages로 구성하고 chat template을 적용하여 텍스트로 변환한다.
- **Step 4 — LoRA 학습**
  - **TODO 3**: LoRA Config 작성 *(연결: LoRA / 하이퍼파라미터 설정)*
  - **1줄 요약**: r, lora_alpha, target_modules 등 LoRA 핵심 파라미터를 설정한다.
  - **TODO 4**: SFTConfig 설정 *(연결: SFTTrainer / 학습 설정)*
  - **1줄 요약**: batch_size, gradient_accumulation_steps 등 학습 설정을 구성한다.

### 주의사항
- 데이터셋의 토큰 길이가 모델의 `max_position_embeddings`를 초과하면 truncation이 발생합니다.
- GPU 메모리 부족 시 `per_device_train_batch_size`를 줄이거나 `gradient_checkpointing`을 활성화하세요.

---

# Step 1: 데이터셋 불러오기 및 EDA

ko_text2sql 데이터셋을 불러오고 EDA를 수행합니다.

### Setup: 라이브러리 설치

In [ ]:
%%capture
# 로컬 환경용 설치 (RTX 5060ti, 16GB VRAM)
%pip install "transformers>=4.56.0" "peft>=0.17.0" "trl>=0.22.0"
%pip install "datasets>=4.0.0" "accelerate>=1.0.0"
%pip install "numpy>=2.0.0" "pandas>=2.0.0"
%pip install unsloth

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import random

# 시드 설정 (재현성)
random.seed(1234)
np.random.seed(1234)
torch.manual_seed(1234)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1234)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

### Concept Check: 실습에서 과제로의 전환

이 과제는 **실습에서 배운 LoRA 개념**을 다른 태스크와 프레임워크에 적용합니다.

| 구분 | 실습 | 과제 |
|------|------|------|
| **프레임워크** | Unsloth (최적화) | HuggingFace PEFT (표준) |
| **태스크** | 체스 기보 예측 | Text-to-SQL |
| **양자화** | QLoRA (4-bit NF4) | LoRA (FP16) |

**실습에서 배운 핵심 (과제에서 활용):**
1. LoRA 원리: 저차원 행렬 분해 (r, lora_alpha)
2. Chat Template: system/user/assistant 역할 구분
3. SFTTrainer: 파인튜닝 파이프라인 자동화

> 프레임워크와 태스크는 다르지만, **LoRA의 핵심 원리는 동일**합니다!

### Concept Check: EDA의 중요성

NLP 태스크에서 EDA(탐색적 데이터 분석)는 모델 학습 전 필수 단계입니다.

**주요 체크 항목:**
1. **Sequence Length**: 입력 텍스트의 최대 길이 확인 → max_seq_length 설정에 영향
2. **Language**: 데이터의 언어 확인 → 적절한 모델 선택
3. **Domain**: 데이터의 도메인 특성 파악 → 전처리 전략 수립

ko_text2sql 데이터셋의 특성:
- Language: 한국어 질문 + SQL 코드
- Domain: 다양한 테이블 스키마에 대한 SQL 쿼리

### Guided Build: 데이터셋 로드

In [ ]:
from datasets import load_dataset
import pandas as pd

# 데이터셋 로드
text_to_sql = load_dataset("shangrilar/ko_text2sql", data_dir="origin")
train_df = text_to_sql["train"].to_pandas()

print(f"데이터셋 크기: {len(train_df)}")
train_df.head()

In [ ]:
# 컬럼 정보 확인
train_df.info()

In [ ]:
# 샘플 데이터 확인
for i, row in train_df.iterrows():
    print("=" * 50)
    for k, v in row.items():
        print(f"{k}: {v[:200] if isinstance(v, str) and len(v) > 200 else v}")
    break

### TODO 1: max_sequence_length 확인

- **요구사항**: 각 컬럼(context, question, answer)의 최대 문자열 길이를 계산하세요.
- **입력**: train_df (데이터프레임)
- **출력**: 각 컬럼의 최대 길이 리스트
- **힌트**: len() 함수와 max() 함수를 사용하세요.

In [ ]:
# TODO: max_sequence_length 확인

# 필요한 컬럼만 선택
columns = ["context", "question", "answer"]
train_data = train_df[columns]

# 각 컬럼의 최대 문자열 길이 계산
max_sequence = [0, 0, 0]
for i, row in train_data.iterrows():
    for idx, (k, v) in enumerate(row.items()):
        max_sequence[idx] = max(max_sequence[idx], len(str(v)))

print(f"컬럼별 최대 문자열 길이:")
for col, length in zip(columns, max_sequence):
    print(f"  - {col}: {length}")

---

# Step 2: 모델/토크나이저 로드

HuggingFace 모델과 토크나이저를 로드합니다.

### Concept Check: HuggingFace 모델 로드

HuggingFace Transformers 라이브러리를 사용하여 사전학습된 모델을 로드합니다.

**주요 설정:**
- `AutoModelForCausalLM`: 언어 생성 모델 로드
- `AutoTokenizer`: 해당 모델의 토크나이저 로드
- `device_map={"": 0}`: 단일 GPU에 전체 모델 로드 (LoRA 학습 시 device 불일치 방지)

**max_position_embeddings:**
- 모델이 처리할 수 있는 최대 토큰 수
- 입력 데이터의 토큰 길이가 이 값을 초과하면 안 됨

> **주의**: `device_map="auto"`는 모델을 GPU/CPU에 자동 분산하여 일부 파라미터가 CPU에 남을 수 있습니다. LoRA 학습 시 모든 파라미터가 같은 device에 있어야 하므로, 단일 GPU 환경에서는 `{"": 0}`을 사용합니다.

### Guided Build: 모델/토크나이저 로드

In [ ]:
# Qwen2.5는 오픈 모델이므로 별도 인증 불필요
# 다른 gated 모델 사용 시 아래 주석 해제
# from huggingface_hub import login
# login(token="your_hf_token")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# 모델 선택 (16GB VRAM에 적합한 오픈 모델)
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 모델 로드 (단일 GPU에 전체 모델 배치)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map={"": 0}
)

print(f"모델 로드 완료: {model_name}")

### Concept Check: 비트 정밀도와 LoRA vs QLoRA

실습: **QLoRA (4-bit NF4)** / 과제: **표준 LoRA (FP16)**

| 정밀도 | 파라미터당 메모리 | 1B 모델 | 사용 시점 |
|--------|-----------------|--------|----------|
| FP32 | 4 bytes | 4 GB | Optimizer states |
| **FP16** | 2 bytes | 2 GB | **이 과제** |
| NF4 | 0.5 bytes | 0.5 GB | 실습 (QLoRA) |

**왜 과제에서는 FP16?**
- Qwen2.5 1.5B는 FP16에서도 16GB VRAM으로 학습 가능
- 표준 HuggingFace PEFT 사용법 학습이 목표

> QLoRA의 "Q"(양자화)를 제외하면 **LoRA 원리는 동일**!

In [ ]:
# 모델의 max_sequence_length 확인
max_sequence_length = model.config.max_position_embeddings

# 토큰화된 길이 확인
max_token_sequence = [0, 0, 0]
for i, row in train_data.iterrows():
    for idx, (k, v) in enumerate(row.items()):
        token_length = len(tokenizer(str(v))["input_ids"])
        max_token_sequence[idx] = max(max_token_sequence[idx], token_length)

print(f"모델 max_position_embeddings: {max_sequence_length}")
print(f"데이터 최대 토큰 길이: {max_token_sequence}")
print(f"총 입력 토큰 길이 예상: {sum(max_token_sequence[:2])} (context + question)")

---

# Step 3: 데이터 전처리

chat template을 적용하고 label masking을 설정합니다.

### Concept Check: Chat Template

LLM은 특정 형식의 입력을 기대합니다. `apply_chat_template`을 사용하여
messages 리스트를 모델이 이해하는 형식으로 변환합니다.

**Qwen2.5 Chat Template:**
```
<|im_start|>system
시스템 프롬프트<|im_end|>
<|im_start|>user
사용자 질문<|im_end|>
<|im_start|>assistant
모델 응답<|im_end|>
```

> 대부분의 최신 LLM은 ChatML 형식(`<|im_start|>`, `<|im_end|>`)을 사용합니다.

In [ ]:
# Chat template 확인
if tokenizer.chat_template:
    print("=== Chat Template 사용 가능 ===")
    print(tokenizer.chat_template)
else:
    print("=== Chat Template 없음 ===")

### Concept Check: Label Masking

학습 시 **instruction 부분은 학습에서 제외**하고 **응답(assistant)만 학습**합니다.

**원리:**
- labels에서 instruction 부분을 `-100`으로 설정
- PyTorch의 CrossEntropyLoss는 `-100`인 위치를 자동으로 무시함

```
input_ids: [instruction_tokens] + [response_tokens]
labels:    [-100, -100, ...]     + [response_tokens]
```

**구현 방법 — `train_on_responses_only`:**

실습에서 사용한 `train_on_responses_only`가 위의 `-100` 마스킹을 자동으로 처리해줍니다.
`instruction_part`와 `response_part`에 chat template의 역할 시작 토큰을 지정하면,
해당 토큰을 기준으로 instruction 영역과 response 영역을 구분하여 label masking을 적용합니다.

```python
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",    # 이 토큰 이후 = instruction (학습 제외)
    response_part="<|im_start|>assistant\n",   # 이 토큰 이후 = response (학습 대상)
)
```

> 실습에서는 Gemma 모델의 `<start_of_turn>user\n` / `<start_of_turn>model\n`을 사용했고,
> 이 과제에서는 Qwen2.5의 ChatML 형식인 `<|im_start|>user\n` / `<|im_start|>assistant\n`을 사용합니다.
> 모델마다 토큰은 다르지만, **사용 방법은 동일**합니다.

### TODO 2: convert_to_conversation 함수 작성

- **요구사항**: 데이터프레임의 각 행을 chat template이 적용된 텍스트로 변환하세요.
- **입력**: examples (데이터셋 딕셔너리)
- **출력**: {"text": [chat template이 적용된 문자열들]}
- **힌트**: system_prompt에 SQL 생성 지시, user에 스키마+질문, assistant에 SQL 쿼리. `tokenizer.apply_chat_template`으로 최종 텍스트 변환

In [ ]:
# TODO: convert_to_conversation 함수 작성

import datasets

# 시스템 프롬프트 (SQL 생성 지시)
system_prompt = """You are a text to SQL query translator. Users will ask you questions in Korean and you will generate a SQL query based on the provided SCHEMA."""

# 사용자 프롬프트 템플릿
user_prompt = """Given the <USER_QUERY> and the <SCHEMA>, generate the corresponding SQL command to retrieve the desired data, considering the query's syntax, semantics, and schema constraints.

<SCHEMA>
{context}
</SCHEMA>

<USER_QUERY>
{question}
</USER_QUERY>"""

def convert_to_conversation(examples):
    """데이터를 chat template이 적용된 텍스트로 변환합니다."""
    train_data = []
    for i in range(len(examples)):
        messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt.format(
                    question=examples["question"][i],
                    context=examples["context"][i]
                )
            },
            {
                "role": "assistant",
                "content": examples["answer"][i]
            }
        ]
        # chat template을 적용하여 모델 입력 형식의 텍스트로 변환
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        train_data.append({"text": text})
    return train_data

# 데이터셋 변환
train_data_list = convert_to_conversation(train_data)
train_dataset = datasets.Dataset.from_list(train_data_list)

print("변환된 샘플:")
print(train_dataset["text"][0])

---

# Step 4: LoRA 학습

LoRA Config를 설정하고 SFTTrainer로 학습합니다.

### Concept Check: LoRA Config 주요 설정

| 파라미터 | 설명 | 이 과제에서의 값 |
|----------|------|------------------|
| r | rank (저차원 행렬 크기) | 16 |
| lora_alpha | 스케일링 값 | 32 (r × 2) |
| lora_dropout | 드롭아웃 비율 | 0.0 |
| target_modules | LoRA 적용 레이어 | Attention + MLP |
| task_type | 태스크 유형 | CAUSAL_LM |

### TODO 3: LoRA Config 작성

- **요구사항**: 실습에서 배운 LoRA 개념을 활용하여 LoRA Config를 작성하세요.
- **입력**: 없음
- **출력**: peft_config 객체
- **힌트**: r=16, lora_alpha=32, target_modules는 Attention + MLP 레이어

In [ ]:
# TODO: LoRA Config 작성

from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

print("LoRA Config 설정 완료!")
print(f"  - r: {peft_config.r}")
print(f"  - lora_alpha: {peft_config.lora_alpha}")
print(f"  - target_modules: {peft_config.target_modules}")

### TODO 4: SFTConfig 설정

- **요구사항**: 학습 설정을 구성하세요.
- **입력**: 없음
- **출력**: train_cfg 객체
- **힌트**: 16GB VRAM에 맞게 batch_size 조절

In [ ]:
# TODO 4: SFTConfig 설정

from trl import SFTConfig, SFTTrainer

train_cfg = SFTConfig(
    output_dir="outputs-text2sql",
    per_device_train_batch_size=1,      # 16GB VRAM에 맞게 조절
    gradient_accumulation_steps=4,       # 실효 배치 크기 = 1 × 4 = 4
    learning_rate=5e-5,
    max_steps=100,                       # 빠른 실습을 위해 100 step만
    logging_steps=10,
    fp16=False,                           # 16bit 학습으로 메모리 절약
    bf16=True,
    optim="adamw_torch",                 # 표준 PyTorch AdamW
    report_to="none",
)

print("SFTConfig 설정 완료!")

### Guided Build: Trainer 설정 및 학습

In [ ]:
# Trainer 설정 (train_cfg에서 설정한 값을 참조)
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset.select(range(min(train_cfg.max_steps * train_cfg.per_device_train_batch_size, len(train_dataset)))),
    peft_config=peft_config,
    args=train_cfg,
)

# Response만 학습하도록 설정 (실습과 동일한 방식)
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print("Trainer 설정 완료!")

In [ ]:
# 학습 시작
print("🚀 학습 시작!")
trainer_stats = trainer.train()
print(f"\n✅ 학습 완료!")
print(f"  - 총 학습 시간: {trainer_stats.metrics['train_runtime']:.2f}초")
print(f"  - 최종 loss: {trainer_stats.metrics['train_loss']:.4f}")

### Concept Check: 파라미터 효율성 검증

실습에서 배운 "1% 미만 파라미터만 학습"하는 LoRA의 효율성을 확인합니다.

In [ ]:
# 학습된 파라미터 수 확인
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_percentage = (trainable_params / total_params) * 100

print(f"\nLoRA 효율성 요약:")
print(f"- 총 파라미터: {total_params:,}")
print(f"- 학습 파라미터: {trainable_params:,}")
print(f"- 학습 비율: {trainable_percentage:.2f}%")
print(f"\n전체의 {trainable_percentage:.2f}%만 학습 → 실습에서 배운 PEFT 효율성 확인!")

In [ ]:
# 테스트 데이터 준비
test_idx = len(train_df) - 1

test_input = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt.format(
        question=train_df["question"][test_idx],
        context=train_df["context"][test_idx]
    )},
]

print(f"정답: {train_df['answer'][test_idx]}")
print("\n" + "=" * 50)

### Test: 학습 결과 테스트

In [ ]:
# 추론
# [중요] 모델을 평가 모드로 전환 (Dropout 등 비활성화)
model.eval()

inputs = tokenizer.apply_chat_template(
    test_input, add_generation_prompt=True, return_dict=True, return_tensors="pt"
)

with torch.no_grad():
    output_ids = model.generate(
        **inputs.to("cuda"),
        max_new_tokens=256,
        stop_strings=["<|im_end|>", "<|endoftext|>"],
        tokenizer=tokenizer
    )

input_prompt = tokenizer.apply_chat_template(test_input, add_generation_prompt=True, tokenize=False)
response = tokenizer.batch_decode(output_ids)[0][len(input_prompt) - 1:]

print(f"모델 응답:\n{response}")

---
## 트러블슈팅 가이드

실습/과제 진행 중 자주 발생하는 오류와 해결 방법입니다.

### GPU / CUDA 관련

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `CUDA out of memory` | GPU 메모리 부족 | 커널 재시작 후 불필요한 모델 제거, `torch.cuda.empty_cache()` |
| `CUDA not available` | CUDA 드라이버 미설치 | `nvidia-smi` 확인 후 드라이버 재설치 |
| `bitsandbytes` 설치 오류 | CUDA 버전 불일치 | `pip install bitsandbytes --prefer-binary` |

### 모델 관련

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `OSError: Can't load tokenizer` | 모델 접근 권한 없음 | HuggingFace 토큰 설정 확인 (`huggingface-cli login`) |
| 모델 다운로드 중단 | 네트워크 불안정 | `resume_download=True` 옵션 사용 |
| `OutOfMemoryError` (시스템 RAM) | 시스템 메모리 부족 | 다른 프로세스 종료 또는 배치 크기 축소 |

### 패키지 관련

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `ModuleNotFoundError` | 패키지 미설치 | Step 1의 `%pip install` 셀 재실행 |
| `ImportError: cannot import name` | 버전 불일치 | `%pip install --upgrade <패키지>` |

### 일반

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `NameError: name 'xxx' is not defined` | 이전 셀 미실행 | Step 1부터 순서대로 재실행 |
| 학습 시 loss가 줄지 않음 | 학습률/에폭 설정 문제 | `learning_rate`, `num_train_epochs` 조정 |


---

## Open-ended Mission

1. 다른 r 값(8, 16, 32)을 적용하여 학습 결과 비교해보기
2. 다른 target_modules 조합으로 학습해보기
3. 학습된 모델을 merge하여 추론 속도 비교해보기

---

## 학생용 자가 체크리스트

- [ ] 모든 TODO를 완료했는가?
- [ ] 코드가 에러 없이 실행되는가?
- [ ] LoRA 하이퍼파라미터의 의미를 설명할 수 있는가?
- [ ] 학습 후 모델이 정상적으로 추론되는가?
- [ ] chat template과 label masking의 역할을 이해했는가?

---

### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 삼성 청년 SW아카데미의 컨텐츠 자산으로, 보안서약서에 의거하여 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.